# Limpieza y Transformación (ETL) del año 2020
Usando el dataframe ya en limpio del año 2019 decidi contraponer el año 2020 y 2021 para luego al final, cuando cree visualizaciones, poder tener un contexto de pre pandemia, pandemia en si y post pandemia para medir.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

path_clean = "../data/clean/"
path_raw = "../data/raw/"

df_2019_clean = pd.read_csv(
    path_clean + "historico_2019_clean.csv",
    sep=",",
    encoding="utf-8-sig"
)

schema_2019 = df_2019_clean.columns.tolist()

print("Columnas schema 2019:", len(schema_2019))
schema_2019


Columnas schema 2019: 17


['periodo',
 'fecha',
 'desde',
 'hasta',
 'linea',
 'molinete',
 'estacion',
 'pax_pagos',
 'pax_pases_pagos',
 'pax_franq',
 'total',
 'hora_desde',
 'hora_hasta',
 'dia_semana',
 'mes',
 'dia_mes',
 'es_fin_semana']

# IMPORTANTE: DATA SCHEMA
Opte por usar el df limpio del 2019 como esquema inicial para respetar la estructura de datos presentada en aquel archivo. Tanto en este file como en el próximo a crear , año 2021, tendran que respetar las columnas de datos propuestas por el 2019 y en caso de tener inconsistencias, completar como lo amerite.


In [2]:
#Carga datos crudos

df_2020 = pd.read_csv(path_raw + "historico_2020.csv")

print(df_2020.shape)
df_2020.head(2)


(5781006, 10)


,FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,pax_pagos,pax_pases_pagos,pax_franq,pax_TOTAL
0,01/01/2020,08:00:00,08:15:00,LineaA,LineaA_Acoyte_N_Turn01,Acoyte,1.0,0.0,0.0,1.0
1,01/01/2020,08:00:00,08:15:00,LineaA,LineaA_Carabobo_E_Turn02,Carabobo,6.0,0.0,0.0,6.0


In [3]:
cols_2020 = set(df_2020.columns)
cols_2019 = set(schema_2019)

faltan_en_2020 = sorted(list(cols_2019 - cols_2020))
sobran_en_2020 = sorted(list(cols_2020 - cols_2019))

print("Faltan en 2020:", faltan_en_2020)
print("Sobran en 2020:", sobran_en_2020)


Faltan en 2020: ['desde', 'dia_mes', 'dia_semana', 'es_fin_semana', 'estacion', 'fecha', 'hasta', 'hora_desde', 'hora_hasta', 'linea', 'mes', 'molinete', 'periodo', 'total']
Sobran en 2020: ['DESDE', 'ESTACION', 'FECHA', 'HASTA', 'LINEA', 'MOLINETE', 'pax_TOTAL']


### Alineación del esquema 2020 con el dataset de referencia 2019

Luego de estandarizar los nombres de columnas del dataset 2020, hice una
comparación contra el esquema de referencia definido a partir del dataset limpio
de 2019.

El análisis muestra que:
- No existen columnas adicionales en el dataset 2020 que no estén presentes en
  el esquema de referencia.
- Las únicas columnas ausentes corresponden a variables derivadas
  (`periodo`, `mes`, `dia_mes`, `dia_semana`, `es_fin_semana`,
  `hora_desde`, `hora_hasta`), las cuales no forman parte del dataset crudo y
  deben ser generadas durante la etapa de transformación.

Este resultado confirma la compatibilidad estructural entre los datasets y
valida el uso de un proceso ETL específico para el año 2020, orientado a la
creación de variables derivadas y a la alineación final del esquema, garantizando
consistencia y comparabilidad temporal con el período pre-pandemia.


In [4]:
rename_map = {
    "FECHA": "fecha",
    "DESDE": "desde",
    "HASTA": "hasta",
    "LINEA": "linea",
    "MOLINETE": "molinete",
    "ESTACION": "estacion",
    "pax_TOTAL": "total",
}

df_2020 = df_2020.rename(columns=rename_map)
df_2020.columns

Index(['fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion', 'pax_pagos', 'pax_pases_pagos', 'pax_franq', 'total'], dtype='object')

In [5]:
df_2020.isna().sum()


fecha              1836
desde              1836
hasta              1836
linea              1836
molinete           1836
estacion           1836
pax_pagos          1836
pax_pases_pagos    1836
pax_franq          1836
total              1836
dtype: int64

### Auditoría de valores faltantes (2020)

Se detectó una proporción baja de valores faltantes (~0.032%) y, de forma consistente, el porcentaje es idéntico en todas las columnas.  
Esto sugiere la existencia de registros incompletos (filas con múltiples campos faltantes simultáneamente), más que un problema aislado de una variable específica.  
Se decide conservar estos registros y manejar los nulos mediante tipados que soporten valores faltantes y reglas robustas de transformación, evitando eliminar datos.


In [6]:
df_2020.loc[df_2020["fecha"].isna(), ["fecha","desde","hasta","linea","estacion"]].head(5)

,fecha,desde,hasta,linea,estacion
3917825,NaN,NaN,NaN,NaN,NaN
3917826,NaN,NaN,NaN,NaN,NaN
3917827,NaN,NaN,NaN,NaN,NaN
3917828,NaN,NaN,NaN,NaN,NaN
3917829,NaN,NaN,NaN,NaN,NaN


In [7]:
df_2020.loc[df_2020["fecha"].notna(), ["fecha","desde","hasta","linea","estacion"]].head(5)


,fecha,desde,hasta,linea,estacion
0,01/01/2020,08:00:00,08:15:00,LineaA,Acoyte
1,01/01/2020,08:00:00,08:15:00,LineaA,Carabobo
2,01/01/2020,08:00:00,08:15:00,LineaA,Castro Barros
3,01/01/2020,08:00:00,08:15:00,LineaA,Castro Barros
4,01/01/2020,08:00:00,08:15:00,LineaA,Congreso


In [8]:
# 1) eliminar filas donde TODAS las columnas son NaN
df_2020 = df_2020.dropna(how="all")


In [9]:
# 2) chequeo: nulos por columna
df_2020.isna().sum().head(10), df_2020.shape

(fecha              0
 desde              0
 hasta              0
 linea              0
 molinete           0
 estacion           0
 pax_pagos          0
 pax_pases_pagos    0
 pax_franq          0
 total              0
 dtype: int64,
 (5779170, 10))

In [10]:
# strings
df_2020["desde"] = df_2020["desde"].astype("string").str.strip().replace("", pd.NA)
df_2020["hasta"] = df_2020["hasta"].astype("string").str.strip().replace("", pd.NA)

df_2020["linea"] = df_2020["linea"].astype("string").str.strip().str.upper()
df_2020["estacion"] = df_2020["estacion"].astype("string").str.strip().str.title()
df_2020["molinete"] = df_2020["molinete"].astype("string").str.strip().str.upper()

# horas desde string HH:MM:SS
df_2020["hora_desde"] = pd.to_numeric(
    df_2020["desde"].str.split(":").str[0],
    errors="coerce"
).astype("Int8")

df_2020["hora_hasta"] = pd.to_numeric(
    df_2020["hasta"].str.split(":").str[0],
    errors="coerce"
).astype("Int8")


In [11]:
s = df_2020["fecha"].astype("string").str.strip()

# Parse principal: ISO YYYY-MM-DD
dt_iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")

# (Opcional) fallback por si hubiera mezcla rara
dt_fallback = pd.to_datetime(s, errors="coerce")

df_2020["fecha"] = dt_iso.fillna(dt_fallback)

df_2020["fecha"].isna().sum()



np.int64(1723320)

In [12]:
df_2020 = df_2020.dropna(how="all")

In [13]:
df_2020.shape

(5779170, 12)

In [14]:
df_2020.isna().sum().head(10)

fecha              1723320
desde                    0
hasta                    0
linea                    0
molinete                 0
estacion                 0
pax_pagos                0
pax_pases_pagos          0
pax_franq                0
total                    0
dtype: int64

In [15]:
df_2020["fecha"].isna().sum()


np.int64(1723320)

In [16]:
df_2020.loc[df_2020["fecha"].isna(), ["fecha","desde","hasta","linea","estacion","pax_pagos"]].head(10)


,fecha,desde,hasta,linea,estacion,pax_pagos
226752,NaT,01:30:00,01:45:00,LINEAC,Constitucion,2.0
226753,NaT,04:00:00,04:15:00,LINEAA,Carabobo,0.0
226754,NaT,04:00:00,04:15:00,LINEAA,Acoyte,0.0
226755,NaT,04:15:00,04:30:00,LINEAA,Carabobo,0.0
226756,NaT,04:15:00,04:30:00,LINEAB,Rosas,0.0
226757,NaT,04:30:00,04:45:00,LINEAA,Plaza Miserere,0.0
226758,NaT,05:00:00,05:15:00,LINEAA,Castro Barros,0.0
226759,NaT,05:00:00,05:15:00,LINEAA,Plaza Miserere,0.0
226760,NaT,05:00:00,05:15:00,LINEAA,Plaza Miserere,0.0
226761,NaT,05:00:00,05:15:00,LINEAA,Puan,0.0


### IMPORTANTE

Durante el análisis del dataset 2020 se identificó un volumen significativo de
registros con información operativa válida (franja horaria, línea, estación y
conteo de pasajeros) pero sin fecha calendario asociada. Estos registros no
representan errores de carga sino datos incompletos desde el origen.

Se decidió conservar dichos registros en el dataset final y permitir valores
nulos en las variables temporales derivadas, excluyéndolos únicamente de los
análisis basados en tiempo. De esta manera se preserva la información real del
sistema sin introducir sesgos en las visualizaciones temporales.


In [17]:
#Vuelvo fecha a string como en el 2019
df_2020["fecha"] = df_2020["fecha"].dt.strftime("%Y-%m-%d")


In [18]:
df_2020.head(2)

,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,hora_desde,hora_hasta
0,2020-01-01,08:00:00,08:15:00,LINEAA,LINEAA_ACOYTE_N_TURN01,Acoyte,1.0,0.0,0.0,1.0,8,8
1,2020-01-01,08:00:00,08:15:00,LINEAA,LINEAA_CARABOBO_E_TURN02,Carabobo,6.0,0.0,0.0,6.0,8,8


In [19]:
#Asegurar tipos numéricos de métricas
for col in ["pax_pagos", "pax_pases_pagos", "pax_franq", "total"]:
    df_2020[col] = (
        pd.to_numeric(df_2020[col], errors="coerce")
        .fillna(0)
        .astype("int64")
    )


In [20]:
#Alinear columnas al schema 2019
for col in schema_2019: #para cada columna del schema 2019
    if col not in df_2020.columns: #si no está en el df_2020
        df_2020[col] = pd.NA #la creo con NA
        
#eliminar columnas sobrantes y ordenar según schema_2019
df_2020 = df_2020[schema_2019]

print("Schema OK: ", df_2020.columns.tolist() == schema_2019)
df_2020.head(3)

Schema OK:  True


,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,hora_desde,hora_hasta,dia_semana,mes,dia_mes,es_fin_semana
0,<NA>,2020-01-01,08:00:00,08:15:00,LINEAA,LINEAA_ACOYTE_N_TURN01,Acoyte,1,0,0,1,8,8,<NA>,<NA>,<NA>,<NA>
1,<NA>,2020-01-01,08:00:00,08:15:00,LINEAA,LINEAA_CARABOBO_E_TURN02,Carabobo,6,0,0,6,8,8,<NA>,<NA>,<NA>,<NA>
2,<NA>,2020-01-01,08:00:00,08:15:00,LINEAA,LINEAA_CBARROS_N_TURN03,Castro Barros,3,0,1,4,8,8,<NA>,<NA>,<NA>,<NA>


In [21]:
df_2019_clean.head(3)

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,hora_desde,hora_hasta,dia_semana,mes,dia_mes,es_fin_semana
0,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LIMA_N_TURN02,Lima,1,0,0,1,8,8,Tuesday,1,1,0
1,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LORIA_N_TURN03,Loria,3,0,0,3,8,8,Tuesday,1,1,0
2,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_Q_HALL_TURN01,Plaza Miserere,3,0,0,3,8,8,Tuesday,1,1,0


En el dataset 2019 la variable `fecha` actúa como una representación del mes,
derivada previamente del campo `periodo`. En los datasets 2020 y posteriores
se dispone de fechas reales a nivel día, por lo que se adopta la lógica inversa:
el campo `periodo` se deriva directamente desde `fecha`, manteniendo coherencia
semántica entre años y mayor fidelidad temporal.


In [23]:
df_2020["periodo"] = (
    df_2020["fecha"].str.slice(0, 4).astype("Int32") * 100 +
    df_2020["fecha"].str.slice(5, 7).astype("Int8")
)


In [24]:
df_2020.head(2)

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,hora_desde,hora_hasta,dia_semana,mes,dia_mes,es_fin_semana
0,202001,2020-01-01,08:00:00,08:15:00,LINEAA,LINEAA_ACOYTE_N_TURN01,Acoyte,1,0,0,1,8,8,<NA>,<NA>,<NA>,<NA>
1,202001,2020-01-01,08:00:00,08:15:00,LINEAA,LINEAA_CARABOBO_E_TURN02,Carabobo,6,0,0,6,8,8,<NA>,<NA>,<NA>,<NA>
